# 06 — Live Analyser — Machine PF30106
### Anomaly Gate only — Three-state monitoring
> State 1: Healthy
> State 4: Approaching Unknown Problem
> State 5: Running With Unknown Problem
> Run 00_Data_Preparation_PF30106.ipynb and 04_Anomaly_Gate_PF30106.ipynb first.

In [11]:
import pandas as pd
import numpy as np
import pickle
import time
import pyodbc
import torch
import torch.nn as nn
import warnings
warnings.filterwarnings('ignore')
try:
    import winsound
    SOUND_AVAILABLE = True
except ImportError:
    SOUND_AVAILABLE = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device: {DEVICE}')

✅ Device: cpu


In [12]:
# ── AZURE SQL CONFIGURATION ──────────────────────────────────────────
AZURE_SERVER   = 'kreseakreiotprdsrv.database.windows.net'
AZURE_DATABASE = 'KRESEAKREIOTPRD'
AZURE_USERNAME = 'IOTAdmin'
AZURE_PASSWORD = 'oKuvodump5JNG7dM'
AZURE_TABLE    = 'MBP_ControllerData'

# ── THRESHOLD OVERRIDES ──────────────────────────────────────────────
# Set to None to use values from ae_thresholds_PF30106.pkl
MANUAL_WARNING_THRESHOLD  = None
MANUAL_CRITICAL_THRESHOLD = None

ELEC_FEATS = [
    'machineVoltageMean', 'machineVoltageMin', 'machineVoltageMax',
    'machineCurrentMean', 'machineCurrentMin', 'machineCurrentMax'
]

print('✅ Configuration loaded.')

✅ Configuration loaded.


#### Load Anomaly Gate

In [13]:
class AnomalyGate(nn.Module):
    def __init__(self, time_steps, num_features, num_vib=60, num_elec=6):
        super().__init__()
        self.time_steps = time_steps
        self.num_vib    = num_vib
        self.num_elec   = num_elec
        self.enc_vib_conv = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2)
        )
        enc_vib_size     = 32 * (num_vib // 2)
        self.enc_lstm    = nn.LSTM(enc_vib_size + num_elec, 32, batch_first=True)
        self.bottleneck  = nn.Linear(32, 16)
        self.dec_lstm    = nn.LSTM(16, 32, batch_first=True)
        self.dec_out     = nn.Linear(32, num_features)

    def encode(self, x):
        batch   = x.size(0)
        vib_in  = x[:, :, :self.num_vib]
        elec_in = x[:, :, self.num_vib:]
        vib_feats = []
        for t in range(self.time_steps):
            vt = vib_in[:, t, :].unsqueeze(1)
            vt = self.enc_vib_conv(vt).view(batch, -1)
            vib_feats.append(vt)
        vib_seq  = torch.stack(vib_feats, dim=1)
        merged   = torch.cat([vib_seq, elec_in], dim=2)
        _, (h,_) = self.enc_lstm(merged)
        return self.bottleneck(h[-1])

    def forward(self, x):
        encoded    = self.encode(x)
        repeated   = encoded.unsqueeze(1).repeat(1, self.time_steps, 1)
        dec_out, _ = self.dec_lstm(repeated)
        return self.dec_out(dec_out)

# Load artifacts
with open('prepared_data_PF30106.pkl', 'rb') as f: d = pickle.load(f)
with open('scaler_PF30106.pkl',        'rb') as f: scaler = pickle.load(f)
with open('ae_thresholds_PF30106.pkl', 'rb') as f: thresholds = pickle.load(f)

TIME_STEPS     = d['TIME_STEPS']
NUM_FEATURES   = d['num_features']
MACHINE_SERIAL = d['MACHINE_SERIAL']

WARNING_THRESHOLD  = MANUAL_WARNING_THRESHOLD  or thresholds['warning']
CRITICAL_THRESHOLD = MANUAL_CRITICAL_THRESHOLD or thresholds['critical']

gate = AnomalyGate(TIME_STEPS, NUM_FEATURES).to(DEVICE)
gate.load_state_dict(torch.load('best_anomaly_gate_PF30106.pt', map_location=DEVICE))
gate.eval()

print(f'✅ Anomaly Gate loaded for machine: {MACHINE_SERIAL}')
print(f'   Warning threshold  : {WARNING_THRESHOLD:.6f}')
print(f'   Critical threshold : {CRITICAL_THRESHOLD:.6f}')

✅ Anomaly Gate loaded for machine: PF30106
   Warning threshold  : 0.874356
   Critical threshold : 1.186382


#### Feature Extraction & Sound Alert

In [14]:
def extract_features(df):
    vib_records = []
    for val in df['machineVibration']:
        vib_dict = {}
        parts = str(val).split(',')
        try:
            for i in range(0, len(parts)-1, 2):
                f_start = int(parts[i])
                vib_dict[f'{f_start}-{f_start+10}Hz'] = int(parts[i+1])
        except: pass
        vib_records.append(vib_dict)
    expected_vib = [f'{i}-{i+10}Hz' for i in range(10, 610, 10)]
    vib_df  = pd.DataFrame(vib_records).reindex(columns=expected_vib, fill_value=0)
    elec_df = df[ELEC_FEATS].ffill().fillna(0).reset_index(drop=True)
    return pd.concat([vib_df.reset_index(drop=True), elec_df], axis=1)

def alert_sound(level):
    if not SOUND_AVAILABLE: return
    if level == 'CRITICAL':
        for _ in range(3): winsound.Beep(1000, 500)
    elif level == 'WARNING':
        for _ in range(2): winsound.Beep(800, 400)

print('✅ Feature extractor ready.')

✅ Feature extractor ready.


#### Live Monitoring Loop
> Three states only:
> State 1 — Healthy
> State 4 — Approaching Unknown Problem
> State 5 — Running With Unknown Problem

In [ ]:
SELECTED_MACHINE = MACHINE_SERIAL

driver   = '{ODBC Driver 17 for SQL Server}'
conn_str = (f'DRIVER={driver};SERVER={AZURE_SERVER};PORT=1433;'
            f'DATABASE={AZURE_DATABASE};UID={AZURE_USERNAME};PWD={AZURE_PASSWORD}')
query    = (f"SELECT TOP {TIME_STEPS} * FROM {AZURE_TABLE} "
            f"WHERE machineSerialNumber = '{SELECTED_MACHINE}' ORDER BY dateTime DESC")

last_processed_time = None

try:
    with pyodbc.connect(conn_str) as conn:
        print(f'\n✅ Connected. Monitoring machine: {SELECTED_MACHINE}')
        print('='*65)

        while True:
            try:
                df_live = pd.read_sql(query, conn)
                df_live = df_live[df_live['machineVibration'].astype(str).str.startswith('10')]

                # Cold start padding
                while len(df_live) < TIME_STEPS:
                    df_live = pd.concat([df_live.iloc[[0]], df_live], ignore_index=True)
                df_live = df_live.iloc[:TIME_STEPS]

                ts = df_live['dateTime'].iloc[0]
                if ts == last_processed_time:
                    time.sleep(1)
                    continue
                last_processed_time = ts

                current_sl_time = pd.Timestamp.now()
                record_utc_time = pd.to_datetime(ts)
                record_sl_time  = record_utc_time + pd.Timedelta(hours=5, minutes=30)

                df_window = df_live.iloc[::-1].reset_index(drop=True)

                # Feature extraction
                features = extract_features(df_window)
                X_scaled = scaler.transform(features.values)
                X_input  = torch.FloatTensor(X_scaled).unsqueeze(0).to(DEVICE)

                # ── ANOMALY GATE ─────────────────────────────────────
                with torch.no_grad():
                    recon    = gate(X_input)
                    ae_error = float(torch.mean(torch.abs(X_input - recon)).cpu())

                # ── STATE DETERMINATION ──────────────────────────────
                if ae_error >= CRITICAL_THRESHOLD:
                    state     = 'RUNNING WITH UNKNOWN PROBLEM'
                    state_num = 5
                    icon      = '🔴'
                    alert_sound('CRITICAL')
                elif ae_error >= WARNING_THRESHOLD:
                    state     = 'APPROACHING UNKNOWN PROBLEM'
                    state_num = 4
                    icon      = '🟡'
                    alert_sound('WARNING')
                else:
                    state     = 'HEALTHY'
                    state_num = 1
                    icon      = '✅'

                # ── OUTPUT ───────────────────────────────────────────
                print(f'\n{"-"*65}')
                print(f'CURRENT TIME (SL)    : {current_sl_time.strftime("%Y-%m-%d %H:%M:%S")}')
                print(f'RECORD TIME (UTC)    : {record_utc_time.strftime("%Y-%m-%d %H:%M:%S")}')
                print(f'RECORD TIME (SL)     : {record_sl_time.strftime("%Y-%m-%d %H:%M:%S")}')
                print(f'MACHINE              : {SELECTED_MACHINE}')
                print(f'STATE                : {icon} {state}')
                print(f'ANOMALY GATE ERROR   : {ae_error:.6f}  [warn={WARNING_THRESHOLD:.4f} crit={CRITICAL_THRESHOLD:.4f}]')
                print('-'*65)

                if state_num == 5:
                    print('ACTION : Stop machine. Contact maintenance supervisor immediately.')
                elif state_num == 4:
                    print('ACTION : Inspect machine. Anomaly detected — gradually rising error.')
                else:
                    print('No maintenance required.')

            except Exception as e:
                print(f'⚠️  Loop error: {e}')
            time.sleep(1)

except KeyboardInterrupt:
    print('\n🛑 Monitoring stopped.')
except Exception as e:
    print(f'\n❌ Connection error: {e}')


✅ Connected. Monitoring machine: PF30106

-----------------------------------------------------------------
CURRENT TIME (SL)    : 2026-03-18 08:44:21
RECORD TIME (UTC)    : 2026-03-17 09:33:21
RECORD TIME (SL)     : 2026-03-17 15:03:21
MACHINE              : PF30106
STATE                : ✅ HEALTHY
ANOMALY GATE ERROR   : 0.654495  [warn=0.8744 crit=1.1864]
-----------------------------------------------------------------
No maintenance required.
